[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLAlchemy, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)

# Engines and URLs &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell builds `scratch/college.db` as the notebook's Setup did, and makes what its worked
examples made: `PrintStatements` on the engine logger, `pool_report`, `count_from_a_thread` and
`college_engine`. Run it first. The tasks do not depend on one another, and the last cell removes
the scratch folder.


In [1]:
import importlib.util
import logging
import secrets
import shutil
import sqlite3
import threading
from pathlib import Path

import sqlalchemy
from sqlalchemy import URL, create_engine, event, make_url, text
from sqlalchemy.exc import IntegrityError, OperationalError
from sqlalchemy.pool import NullPool, StaticPool

SCRATCH = Path("scratch")
shutil.rmtree(SCRATCH, ignore_errors=True)
SCRATCH.mkdir()
DATABASE = SCRATCH / "college.db"

NAMES = [
    "Ana Reyes", "Ben Okafor", "Chloe Martin", "Daniel Kim", "Elena Petrova", "Felix Wagner",
    "Grace Lin", "Hassan Ali", "Isabel Costa", "Jonas Berg", "Keiko Tanaka", "Liam Murphy",
    "Maya Patel", "Noah Andersen", "Olivia Brandt", "Pavel Novak", "Quinn Harper", "Rosa Delgado",
    "Sam Ito", "Tara Nilsen", "Umar Farouk", "Vera Kowalski", "Wes Carter", "Yara Haddad",
    "Aoife O'Brien",
]
PROGRAMS = ["Biology", "Computer Science", "Mathematics", "Psychology", "History"]
STARTS = ["2024-08-26", "2025-01-13", "2025-08-25"]           # the first day of three terms
STUDENTS = [(name, f"{name[0]}{name.split()[-1]}@college.edu".lower().replace("'", ""),
             PROGRAMS[i % len(PROGRAMS)], STARTS[i % len(STARTS)]) for i, name in enumerate(NAMES)]
COURSES = [
    ("BIO-101", "Introduction to Biology", "Biology", 4),
    ("CHE-110", "General Chemistry", "Chemistry", 4),
    ("MAT-120", "Calculus I", "Mathematics", 4),
    ("MAT-121", "Calculus II", "Mathematics", 4),
    ("CSC-101", "Programming I", "Computer Science", 3),
    ("CSC-201", "Data Structures", "Computer Science", 3),
    ("ENG-105", "Composition", "English", 3),
    ("HIS-110", "World History", "History", 3),
    ("PSY-101", "Introduction to Psychology", "Psychology", 3),
    ("STA-200", "Statistics", "Mathematics", 3),
]

build = sqlite3.connect(DATABASE)
build.executescript("""
    CREATE TABLE students (id INTEGER PRIMARY KEY, name TEXT NOT NULL, email TEXT NOT NULL UNIQUE,
                           program TEXT NOT NULL, started_on TEXT NOT NULL);
    CREATE TABLE courses (id INTEGER PRIMARY KEY, code TEXT NOT NULL UNIQUE, title TEXT NOT NULL,
                          department TEXT NOT NULL, credits INTEGER NOT NULL);
""")
build.executemany("INSERT INTO students (name, email, program, started_on) VALUES (?, ?, ?, ?)", STUDENTS)
build.executemany("INSERT INTO courses (code, title, department, credits) VALUES (?, ?, ?, ?)", COURSES)
build.commit()
build.close()

class PrintStatements(logging.Handler):
    """Print what an engine logs, leaving out the time: every statement, and the values sent with it."""

    def emit(self, record):
        if record.msg == "[%s] %r":                  # after a statement: how long it took, then its values
            values = repr(record.args[1])
            if values != "()":
                print("    values:", values)
        else:
            for line in record.getMessage().splitlines():
                print("   ", line.rstrip())


sql_log = logging.getLogger("sqlalchemy.engine.Engine")
sql_log.handlers = [PrintStatements()]              # this handler alone, however often the cell runs
sql_log.propagate = False                           # and no handler above it prints the same lines again


def pool_report(engine):
    """How many of an engine's connections are lent out, and how many wait in its pool."""
    return f"{engine.pool.checkedout()} lent out, {engine.pool.checkedin()} waiting in the pool"


def count_from_a_thread(engine):
    """Count the students on a connection a new thread borrows, and return the count, or the error it raised."""
    outcome = []

    def count():
        try:
            with engine.connect() as conn:
                outcome.append(conn.execute(text("SELECT COUNT(*) FROM students")).scalar_one())
        except Exception as error:                  # an error raised in another thread never reaches the cell
            outcome.append(error)

    worker = threading.Thread(target=count)
    worker.start()
    worker.join()
    return outcome[0]


def college_engine(path=None, echo=False):
    """An engine for the college's database, in a file or in memory, with foreign keys enforced."""
    if path is None:                                # in memory: one connection, and one database, for every thread
        engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"check_same_thread": False}, echo=echo)
    else:
        engine = create_engine(f"sqlite:///{path}", echo=echo)

    @event.listens_for(engine, "connect")
    def enforce_foreign_keys(dbapi_connection, connection_record):
        dbapi_connection.execute("PRAGMA foreign_keys = ON")

    return engine


print("sqlalchemy", sqlalchemy.__version__, "|", DATABASE, "holds", len(STUDENTS), "students")


sqlalchemy 2.0.54 | scratch/college.db holds 25 students


**1.** A MySQL URL, taken apart.


In [2]:
url = make_url("mysql+pymysql://registrar@db.example.edu:3306/college?charset=utf8mb4")

print("backend:      ", url.get_backend_name())
print("driver:       ", url.get_driver_name())
print("host and port:", url.host, url.port)
print("database:     ", url.database)
print("options:      ", dict(url.query))


backend:       mysql
driver:        pymysql
host and port: db.example.edu 3306
database:      college
options:       {'charset': 'utf8mb4'}


`mysql` is the dialect and `pymysql` the driver, which `get_driver_name` reads from the URL without
importing it. `charset=utf8mb4` goes to the driver when the engine connects.


**2.** A SQLite URL built from its parts, and an engine that has not connected yet.


In [3]:
url = URL.create("sqlite", database="scratch/archive/2024.db")
print(url)

archive = create_engine(url)
print("folder there:", Path("scratch/archive").exists(), "| file there:", Path("scratch/archive/2024.db").exists())
archive.dispose()


sqlite:///scratch/archive/2024.db
folder there: False | file there: False


`URL.create` wrote the three slashes of a relative path for itself, and making the engine touched
nothing on disk.


**3.** The SQL log for every engine, through the logger's level.


In [4]:
logging.getLogger("sqlalchemy.engine").setLevel(logging.INFO)

engine = create_engine(f"sqlite:///{DATABASE}")
with engine.connect() as conn:
    print(conn.execute(text("SELECT COUNT(*) FROM courses")).scalar_one(), "courses")

logging.getLogger("sqlalchemy.engine").setLevel(logging.WARNING)
engine.dispose()


    BEGIN (implicit)
    SELECT COUNT(*) FROM courses
10 courses
    ROLLBACK


The engine was made without `echo`, and logged anyway, because `sqlalchemy.engine.Engine` takes its
level from `sqlalchemy.engine` above it. `PrintStatements` printed the lines, and the count came
before the `ROLLBACK` because it was printed inside the block. `WARNING` puts the level back where
it started, so no engine logs its statements after this cell.


**4.** `NullPool`, and whether a connection comes back.


In [5]:
engine = create_engine(f"sqlite:///{DATABASE}", poolclass=NullPool)

with engine.connect() as conn:
    first = conn.connection.dbapi_connection
with engine.connect() as conn:
    print("the same sqlite3 connection:", conn.connection.dbapi_connection is first)
engine.dispose()


the same sqlite3 connection: False


`NullPool` closed the first connection at the end of its block and opened a new one for the second,
where a `QueuePool` lent the same connection twice. For a file, nothing is lost but the time it takes
to open one.


**5.** A SQL function on every connection.


In [6]:
engine = create_engine(f"sqlite:///{DATABASE}")


@event.listens_for(engine, "connect")
def add_initials(dbapi_connection, connection_record):
    dbapi_connection.create_function("initials", 1, lambda name: "".join(part[0] for part in name.split()))


with engine.connect() as conn:
    rows = conn.execute(text("SELECT name, initials(name) FROM students WHERE program = 'Biology' ORDER BY name"))
    for name, initials in rows:
        print(f"{name:<14}", initials)
engine.dispose()


Ana Reyes      AR
Felix Wagner   FW
Keiko Tanaka   KT
Pavel Novak    PN
Umar Farouk    UF


A function made with `create_function` belongs to one `sqlite3` connection, like the foreign keys
pragma, so the `connect` event is where it goes: every connection the pool ever opens knows
`initials`.


**6.** The SQL of a count for every program.


In [7]:
engine = college_engine(DATABASE, echo=True)
with engine.connect() as conn:
    counts = conn.execute(text("SELECT program, COUNT(*) FROM students GROUP BY program ORDER BY program")).all()
print(counts)
engine.dispose()


    BEGIN (implicit)
    SELECT program, COUNT(*) FROM students GROUP BY program ORDER BY program
    ROLLBACK
[('Biology', 5), ('Computer Science', 5), ('History', 5), ('Mathematics', 5), ('Psychology', 5)]


Five programs with five students each. The `connect` event's `PRAGMA` did not appear in the log,
because it ran on the driver's own connection, underneath the engine that does the logging.

Last, remove the scratch folder:


In [8]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Engines and URLs](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlalchemy-deep-dive/02-engines-and-urls.ipynb)  &nbsp;&middot;&nbsp;  [SQLAlchemy, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlalchemy-deep-dive.html)
